<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="300" alt="Skills Network Logo">
    </a>
</p>


# Test Environment for Generative AI classroom labs

This lab provides a test environment for the codes generated using the Generative AI classroom.

Follow the instructions below to set up this environment for further use.


# Setup


### Install required libraries

In case of a requirement of installing certain python libraries for use in your task, you may do so as shown below.


In [1]:
%pip install seaborn
import piplite

await piplite.install(['nbformat', 'plotly'])

### Dataset URL from the GenAI lab
Use the URL provided in the GenAI lab in the cell below. 


In [2]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-Coursera/laptop_pricing_dataset_mod1.csv"


### Downloading the dataset

Execute the following code to download the dataset in to the interface.

> Please note that this step is essential in JupyterLite. If you are using a downloaded version of this notebook and running it on JupyterLabs, then you can skip this step and directly use the URL in pandas.read_csv() function to read the dataset as a dataframe


In [3]:
from pyodide.http import pyfetch

async def download(url, filename):
    response = await pyfetch(url)
    if response.status == 200:
        with open(filename, "wb") as f:
            f.write(await response.bytes())

path = URL

await download(path, "dataset.csv")
file_name  = "dataset.csv"

---


# Test Environment


In [4]:
# Keep appending the code generated to this cell, or add more cells below this to execute in parts
import pandas as pd

def read_csv_with_headers(file_path: str, sep: str = ',', encoding: str = 'utf-8') -> pd.DataFrame:
    """
    Reads a CSV file into a DataFrame.
    Assumes the first row of the file contains the column headers.
    """
    df = pd.read_csv(file_path, sep=sep, encoding=encoding, header=0)
    return df

# Example usage
if __name__ == "__main__":
    path = "dataset.csv"  # replace with your actual file path
    df = read_csv_with_headers(path)
    print(f"Loaded {len(df)} rows and {len(df.columns)} columns.")
    print(df.head())

<ipython-input-4-ab3c9b32fc0d>:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


Loaded 238 rows and 13 columns.
   Unnamed: 0 Manufacturer  Category     Screen  GPU  OS  CPU_core  \
0           0         Acer         4  IPS Panel    2   1         5   
1           1         Dell         3    Full HD    1   1         3   
2           2         Dell         3    Full HD    1   1         7   
3           3         Dell         4  IPS Panel    2   1         5   
4           4           HP         4    Full HD    2   1         7   

   Screen_Size_cm  CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  
0          35.560            1.6       8             256       1.60    978  
1          39.624            2.0       4             256       2.20    634  
2          39.624            2.7       8             256       2.20    946  
3          33.782            1.6       8             128       1.22   1244  
4          39.624            1.8       8             256       1.91    837  


In [11]:
def missing_value_summary(df: pd.DataFrame):
    """
    Returns:
      - cols_with_missing: list[str] of column names that contain at least one missing value
      - counts: pd.Series with missing value counts per column (index = column name)
      - report: pd.DataFrame with columns ['column','missing_count','missing_pct'] for columns with missing values
    """
    total_rows = len(df)
    counts = df.isna().sum()
    cols_with_missing = counts[counts > 0].index.tolist()

    report = (
        counts[counts > 0]
        .reset_index()
        .rename(columns={'index': 'column', 0: 'missing_count'})
    )
    report['missing_pct'] = (report['missing_count'] / total_rows) * 100
    report = report.sort_values('missing_count', ascending=False).reset_index(drop=True)

    return cols_with_missing, counts, report
df = pd.read_csv('dataset.csv')  # your DataFrame
cols, counts, report = missing_value_summary(df)
print("Columns with missing data:", cols)
print("\nMissing counts per column:\n", counts)
print("\nDetailed report:\n", report)

Columns with missing data: ['Screen_Size_cm', 'Weight_kg']

Missing counts per column:
 Unnamed: 0        0
Manufacturer      0
Category          0
Screen            0
GPU               0
OS                0
CPU_core          0
Screen_Size_cm    4
CPU_frequency     0
RAM_GB            0
Storage_GB_SSD    0
Weight_kg         5
Price             0
dtype: int64

Detailed report:
            column  missing_count  missing_pct
0       Weight_kg              5     2.100840
1  Screen_Size_cm              4     1.680672


In [13]:
def impute_missing_guidelines(df: pd.DataFrame) -> pd.DataFrame:
    """
    Imputes missing values according to:
      - Screen_Size_cm (categorical): fill with the most frequent value (mode)
      - Weight_kg (continuous): fill with the mean value
    Returns a new DataFrame with imputed values.
    """
    out = df.copy()

    # Impute Screen_Size_cm with the mode (most frequent value)
    if 'Screen_Size_cm' in out.columns:
        series = out['Screen_Size_cm']
        modes = series.mode()
        if not modes.empty:
            out['Screen_Size_cm'] = series.fillna(modes.iloc[0])

    # Impute Weight_kg with the mean
    if 'Weight_kg' in out.columns:
        series = out['Weight_kg']
        mean_val = series.mean()
        if pd.notna(mean_val):
            out['Weight_kg'] = series.fillna(mean_val)

    return out
df = pd.read_csv('dataset.csv')
df_imputed = impute_missing_guidelines(df)
print(df_imputed.head())

   Unnamed: 0 Manufacturer  Category     Screen  GPU  OS  CPU_core  \
0           0         Acer         4  IPS Panel    2   1         5   
1           1         Dell         3    Full HD    1   1         3   
2           2         Dell         3    Full HD    1   1         7   
3           3         Dell         4  IPS Panel    2   1         5   
4           4           HP         4    Full HD    2   1         7   

   Screen_Size_cm  CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  
0          35.560            1.6       8             256       1.60    978  
1          39.624            2.0       4             256       2.20    634  
2          39.624            2.7       8             256       2.20    946  
3          33.782            1.6       8             128       1.22   1244  
4          39.624            1.8       8             256       1.91    837  


In [15]:
def convert_columns_to_float(df: pd.DataFrame, cols=None) -> pd.DataFrame:
    """
    Convert specified columns to float using coercion for non-numeric values.
    If a column is missing, it is skipped.
    """
    if cols is None:
        cols = ["Screen_Size_cm", "Weight_kg"]
    df_out = df.copy()
    for c in cols:
        if c in df_out.columns:
            df_out[c] = pd.to_numeric(df_out[c], errors="coerce")
    return df_out

df = pd.read_csv("dataset.csv")
df_float = convert_columns_to_float(df)
print(df_float.dtypes)

Unnamed: 0          int64
Manufacturer       object
Category            int64
Screen             object
GPU                 int64
OS                  int64
CPU_core            int64
Screen_Size_cm    float64
CPU_frequency     float64
RAM_GB              int64
Storage_GB_SSD      int64
Weight_kg         float64
Price               int64
dtype: object


In [16]:
def convert_dims_and_rename(df: pd.DataFrame) -> pd.DataFrame:
    """
    - Convert Screen_Size_cm to inches and rename to Screen_Size_inch
    - Convert Weight_kg to pounds and rename to Weight_pounds
    Missing columns are handled gracefully.
    """
    out = df.copy()

    # Convert Screen_Size_cm to Screen_Size_inch (cm -> inches)
    if 'Screen_Size_cm' in out.columns:
        out['Screen_Size_inch'] = out['Screen_Size_cm'] / 2.54
        out = out.drop(columns=['Screen_Size_cm'])

    # Convert Weight_kg to Weight_pounds (kg -> pounds)
    if 'Weight_kg' in out.columns:
        out['Weight_pounds'] = out['Weight_kg'] * 2.20462262185
        out = out.drop(columns=['Weight_kg'])

    return out

# Example usage:
df = pd.read_csv("dataset.csv")
df_converted = convert_dims_and_rename(df)
print(df_converted.head())

   Unnamed: 0 Manufacturer  Category     Screen  GPU  OS  CPU_core  \
0           0         Acer         4  IPS Panel    2   1         5   
1           1         Dell         3    Full HD    1   1         3   
2           2         Dell         3    Full HD    1   1         7   
3           3         Dell         4  IPS Panel    2   1         5   
4           4           HP         4    Full HD    2   1         7   

   CPU_frequency  RAM_GB  Storage_GB_SSD  Price  Screen_Size_inch  \
0            1.6       8             256    978              14.0   
1            2.0       4             256    634              15.6   
2            2.7       8             256    946              15.6   
3            1.6       8             128   1244              13.3   
4            1.8       8             256    837              15.6   

   Weight_pounds  
0       3.527396  
1       4.850170  
2       4.850170  
3       2.689640  
4       4.210829  


In [17]:
def normalize_cpu_frequency_in_place(df: pd.DataFrame) -> None:
    """
    Normalize the 'CPU_frequency' column by its maximum value.
    - Converts the column to numeric (coercing non-numeric to NaN)
    - Divides by the maximum value (in-place)
    - If the column is missing or the max is NaN or <= 0, no change is made
    """
    if 'CPU_frequency' not in df.columns:
        return

    # Ensure numeric for proper max calculation
    df['CPU_frequency'] = pd.to_numeric(df['CPU_frequency'], errors='coerce')

    max_val = df['CPU_frequency'].max()
    if pd.isna(max_val) or max_val <= 0:
        return  # nothing to scale or invalid max

    df['CPU_frequency'] = df['CPU_frequency'] / max_val

# Example usage:
df = pd.read_csv("dataset.csv")
normalize_cpu_frequency_in_place(df)
print(df['CPU_frequency'].head())

0    0.551724
1    0.689655
2    0.931034
3    0.551724
4    0.620690
Name: CPU_frequency, dtype: float64


In [18]:
import re

def _sanitize_value(val: object) -> str:
    s = str(val).strip()
    s = s.replace(' ', '_').replace('-', '_')
    # keep only alphanumeric and underscore
    s = re.sub(r'[^A-Za-z0-9_]', '', s)
    return s

def convert_screen_to_indicators(df: pd.DataFrame):
    """
    1) Create indicator columns for each unique value in df['Screen'].
       Columns are named Screen_<unique_value>, with simple sanitization.
    2) Append these indicator columns to the original df.
    3) Drop the original 'Screen' column.

    Returns:
      df_out: DataFrame with the original columns (except 'Screen') plus indicators
      df1: DataFrame of the indicator columns (saved as df1)
    """
    df_out = df.copy()
    df1 = pd.DataFrame(index=df_out.index)

    if 'Screen' in df_out.columns:
        uniques = df_out['Screen'].dropna().unique()
        for val in uniques:
            col_name = f"Screen_{_sanitize_value(val)}"
            df1[col_name] = (df_out['Screen'] == val).astype(int)

        # Append indicators to the original DataFrame
        df_out = pd.concat([df_out, df1], axis=1)
        # Drop the original 'Screen' column
        df_out = df_out.drop(columns=['Screen'])

    return df_out, df1

# Example usage:
df_out, df1 = convert_screen_to_indicators(df)
print(df1.head())
print(df_out.head())

   Screen_IPS_Panel  Screen_Full_HD
0                 1               0
1                 0               1
2                 0               1
3                 1               0
4                 0               1
   Unnamed: 0 Manufacturer  Category  GPU  OS  CPU_core  Screen_Size_cm  \
0           0         Acer         4    2   1         5          35.560   
1           1         Dell         3    1   1         3          39.624   
2           2         Dell         3    1   1         7          39.624   
3           3         Dell         4    2   1         5          33.782   
4           4           HP         4    2   1         7          39.624   

   CPU_frequency  RAM_GB  Storage_GB_SSD  Weight_kg  Price  Screen_IPS_Panel  \
0       0.551724       8             256       1.60    978                 1   
1       0.689655       4             256       2.20    634                 0   
2       0.931034       8             256       2.20    946                 0   
3       0.551

In [27]:
from typing import Optional
import requests
import json

def convert_price_usd_to_eur(
    df: pd.DataFrame,
    rate: Optional[float] = None,
    column: str = "Price",
    new_column: Optional[str] = None,
    fetch_rate: bool = False,
    api_url: str = "https://api.exchangerate.host/latest"
) -> pd.DataFrame:
    """
    Convert USD prices to EUR.

    - Coerce the source column to numeric (non-numeric -> NaN).
    - Use a provided rate or fetch the rate from the API if fetch_rate is True.
    - Write results to a new column if new_column is provided; otherwise overwrite the source column.
    - Modify the DataFrame in place and return it for chaining.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")

    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame")

    # Coerce to numeric (non-numeric -> NaN)
    usd_values = pd.to_numeric(df[column], errors="coerce")

    # Determine rate
    if rate is not None:
        if rate <= 0:
            raise ValueError("rate must be positive")
        usd_to_eur = float(rate)
    elif fetch_rate:
        try:
            resp = requests.get(api_url, params={"base": "USD", "symbols": "EUR"}, timeout=10)
            resp.raise_for_status()
            data = resp.json()

            # Handle common error shapes
            if isinstance(data, dict) and data.get("success") is False:
                raise ValueError(
                    "API request failed. Response: " + json.dumps(data)
                )

            rates = data.get("rates") or {}
            eur_rate = rates.get("EUR")

            # Fallback: case-insensitive key match
            if eur_rate is None:
                for k, v in rates.items():
                    if str(k).upper() == "EUR":
                        eur_rate = v
                        break

            if eur_rate is None:
                raise ValueError(
                    "EUR rate not found in API response. Response: "
                    + json.dumps(data)[:300]
                )

            usd_to_eur = float(eur_rate)
            if usd_to_eur <= 0:
                raise ValueError("Fetched rate must be positive.")
        except requests.RequestException as e:
            raise ConnectionError(f"Failed to fetch USD->EUR rate: {e}")
        except ValueError:
            raise
    else:
        raise ValueError("Provide a rate or set fetch_rate=True to fetch the rate from the API.")

    eur_values = usd_values * usd_to_eur

    if new_column:
        df[new_column] = eur_values
    else:
        df[column] = eur_values

    return df

# Example usage:
# Correct DataFrame creation
df = pd.DataFrame({"Price": ["10", 20, None, "abc"]})

# Overwrite the Price column with a provided rate
df_out = convert_price_usd_to_eur(df, rate=0.92)

print(df_out)

   Price
0    9.2
1   18.4
2    NaN
3    NaN


In [28]:
import numpy as np
from typing import Optional

def min_max_normalize_cpu_frequency(
    df: pd.DataFrame,
    column: str = "CPU_frequency",
    new_column: Optional[str] = None
) -> pd.DataFrame:
    """
    Apply min-max normalization to the CPU_frequency column.

    - Non-numeric values are coerced to NaN.
    - Min/Max are computed from non-NaN values.
    - If max != min, normalize: (x - min) / (max - min).
    - If max == min or min/max is NaN, set normalized value to 0.0 for non-NaN entries.
    - NaNs remain NaN in the result.
    - If new_column is provided, write to that column; otherwise overwrite the source column.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame")

    if column not in df.columns:
        raise KeyError(f"Column '{column}' not found in DataFrame")

    # Coerce to numeric
    vals = pd.to_numeric(df[column], errors="coerce")

    minv = vals.min()
    maxv = vals.max()

    # Compute denominator if possible
    denom = None
    if pd.notna(minv) and pd.notna(maxv):
        denom = maxv - minv

    # Normalize
    if denom is not None and denom != 0:
        norm = (vals - minv) / denom
        norm = norm.astype(float)
    else:
        # max == min or invalid min/max: set 0.0 for non-NaN entries
        norm = pd.Series(np.nan, index=vals.index, dtype=float)
        non_nan_idx = vals.notna()
        norm[non_nan_idx] = 0.0

    # Write result
    if new_column:
        df[new_column] = norm
    else:
        df[column] = norm

    return df

# Example usage:
df = pd.DataFrame({"CPU_frequency": ["2.5", 3.0, None, "not a number", 1.5]})
df_norm = min_max_normalize_cpu_frequency(df, new_column="CPU_frequency_norm")
print(df_norm)

  CPU_frequency  CPU_frequency_norm
0           2.5            0.666667
1           3.0            1.000000
2          None                 NaN
3  not a number                 NaN
4           1.5            0.000000


## Authors


[Abhishek Gagneja](https://www.linkedin.com/in/abhishek-gagneja-23051987/)


## Change Log


|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
|-|-|-|-|
|2023-12-10|0.1|Abhishek Gagneja|Initial Draft created|


Copyright © 2023 IBM Corporation. All rights reserved.
